You are given a large NumPy array of size 5000000 initialized with random
values. Compute the following element-wise opera8on: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.

In [ ]:
import numpy as np
from numba import cuda
import time

N = 5000000
x = np.random.rand(N).astype(np.float32)

def cpu_compute(x):
    return x**2 + 3*x + 5

start_cpu = time.time()
y_cpu = cpu_compute(x)
end_cpu = time.time()

cpu_time = end_cpu - start_cpu
print(f"CPU Time: {cpu_time:.6f} seconds")

@cuda.jit
def gpu_compute(x, y):
    idx = cuda.grid(1)
    if idx < x.size:
        y[idx] = x[idx] * x[idx] + 3 * x[idx] + 5

x_device = cuda.to_device(x)
y_device = cuda.device_array_like(x)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

gpu_compute[blocks_per_grid, threads_per_block](x_device, y_device)
cuda.synchronize()

start_gpu = time.time()
gpu_compute[blocks_per_grid, threads_per_block](x_device, y_device)
cuda.synchronize()
end_gpu = time.time()

gpu_time = end_gpu - start_gpu

y_gpu = y_device.copy_to_host()

print(f"GPU Time: {gpu_time:.6f} seconds")

CPU Time: 0.013369 seconds
GPU Time: 0.000791 seconds


Write a function monte_carlo_pi(nsamples) that estimates the value of π by
generating random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).

a. Implement the function in pure Python first and later create a Numba
version.

b. Program a script to compare the execution time for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execution of the Numba function take slightly
longer than the second execution?

In [ ]:
import numpy as np
import random
import time
from numba import njit

def monte_carlo_pi_python(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1.0:
            inside += 1
    return 4.0 * inside / nsamples

@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0
    for i in range(nsamples):
        x = np.random.rand()
        y = np.random.rand()
        if x*x + y*y < 1.0:
            inside += 1
    return 4.0 * inside / nsamples

N = 5_000_000

start_py = time.time()
pi_py = monte_carlo_pi_python(N)
end_py = time.time()
python_time = end_py - start_py

monte_carlo_pi_numba(10)

start_nb = time.time()
pi_nb = monte_carlo_pi_numba(N)
end_nb = time.time()
numba_time = end_nb - start_nb

print(f"Python pi estimate: {pi_py}")
print(f"Python Time: {python_time:.4f} sec")

print(f"Numba pi estimate: {pi_nb}")
print(f"Numba Time: {numba_time:.4f} sec")

print(f"Speedup Factor: {python_time / numba_time:.2f}x")

Python pi estimate: 3.142168
Python Time: 0.7565 sec
Numba pi estimate: 3.1426248
Numba Time: 0.1009 sec
Speedup Factor: 7.50x


You have a 1D NumPy array represen8ng pixel intensities (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize
decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the time difference when the work is automatically split
across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this
function

In [ ]:
import numpy as np
import time
from numba import vectorize

@vectorize
def adjust_brightness(pixel):
    val = int(pixel * 1.2)
    return 255 if val > 255 else val

N = 10_000_000
pixels = np.random.randint(0, 256, N, dtype=np.int64)

start = time.time()
bright_pixels = adjust_brightness(pixels)
end = time.time()
t1 = end - start

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel):
    val = int(pixel * 1.2)
    return 255 if val > 255 else val

start_p = time.time()
bright_pixels_p = adjust_brightness_parallel(pixels)
end_p = time.time()
t2 = end_p - start_p

print("Default time:", t1)
print("Parallel time:", t2)
print("Speedup:", t1 / t2)
print("Same output:", np.array_equal(bright_pixels, bright_pixels_p))

Default time: 0.08492803573608398
Parallel time: 0.02228069305419922
Speedup: 3.8117322261695845
Same output: True


Write Python code to generate synthe8c training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logis8c regression
using the mathematical formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT acceleration

c. Compare correctness and performance.

In [ ]:
import numpy as np
import time
from numba import njit

n, d = 100_000, 10

X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = np.sign(X @ w_true)
y[y == 0] = 1

def train_numpy(X, y):
    w = np.zeros(d)
    yb = (y + 1) / 2
    for _ in range(100):
        z = X @ w
        p = 1 / (1 + np.exp(-z))
        w -= 0.1 * (X.T @ (p - yb)) / n
    return w

@njit
def train_numba(X, y):
    w = np.zeros(d)
    yb = (y + 1) / 2
    for _ in range(100):
        z = np.dot(X, w)
        p = 1 / (1 + np.exp(-z))
        w -= 0.1 * (np.dot(X.T, (p - yb))) / n
    return w

start = time.time()
w1 = train_numpy(X, y)
t1 = time.time() - start

train_numba(X[:10], y[:10])

start = time.time()
w2 = train_numba(X, y)
t2 = time.time() - start

print("NumPy:", t1)
print("Numba:", t2)
print("Speedup:", t1 / t2)


NumPy: 0.6648004055023193
Numba: 0.32584309577941895
Speedup: 2.0402470210765467


Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [ ]:
import numpy as np
from numba import cuda

size = 1024

A = np.random.rand(size, size).astype(np.float32)
B = np.random.rand(size, size).astype(np.float32)

@cuda.jit
def add_matrices(a, b, c):
    i, j = cuda.grid(2)
    if i < a.shape[0] and j < a.shape[1]:
        c[i][j] = a[i][j] + b[i][j]

A_gpu = cuda.to_device(A)
B_gpu = cuda.to_device(B)
C_gpu = cuda.device_array_like(A)

threads_per_block = (16, 16)
blocks_per_grid = (size // 16, size // 16)

add_matrices[blocks_per_grid, threads_per_block](A_gpu, B_gpu, C_gpu)

C = C_gpu.copy_to_host()

